# Additional experiments

Mirrors `README.md`'s run order; each stage checkpoints, so a cell can be rerun to resume. See `PROVENANCE.md` before citing numbers.

In [ ]:
# one-time dependencies beyond the repo dev environment
%pip -q install lightgbm pmlb rdata

In [ ]:
# datasets from canonical sources (network access required)
!python fetch_data.py --all

## Comparison study

In [ ]:
!python bench.py highdim ridge_raw,rf_raw,lgbm_raw,beamfeat,beamfeat_ridge 5 results/highdim_fast.json

In [ ]:
!python bench.py highdim featuretools,openfe 5 results/highdim_constructors.json

`autofeat` runs from its pinned venv as in the original study (`../independent/setup_env.sh`), then:

    python bench.py highdim autofeat 5 results/highdim_autofeat.json

## Search characterisation

In [ ]:
!python depth_ladder.py --seeds 20 --out results/depth_ladder.json

In [ ]:
!python depth_ladder.py --seeds 20 --binary mul,div --out results/depth_ladder_ops_muldiv.json
!python depth_ladder.py --seeds 20 --unary square,abs --out results/depth_ladder_ops_nosqrt.json

In [ ]:
!python scalability.py --p-grid 10,30,100,300,1000 --seeds 5 --out results/scalability.json

## Selector comparison

In [ ]:
!python selector_comparison.py --trials 100 --out results/selector_comparison.json
!python selector_comparison.py --trials 100 --m 100 --k 10 --out results/selector_comparison_m100.json

## Split stability

In [ ]:
!python split_stability.py --splits 30 --out results/split_stability.json

In [ ]:
!python multisplit.py data/tecator.csv fat --splits 20

## Tables and figures

In [ ]:
import json, glob, collections
import numpy as np, pandas as pd

rows = []
for f in glob.glob("results/highdim_*.json"):
    rows += json.load(open(f))
df = pd.DataFrame(rows)
ok = df[df.error.isna()]
summary = (ok.groupby("method")
             .agg(mean_r2=("r2", "mean"), worst=("r2", "min"),
                  neg=("r2", lambda s: int((s < 0).sum())),
                  mean_features=("n_new", "mean"), mean_s=("seconds", "mean"))
             .sort_values("mean_r2", ascending=False).round(3))
errors = (df[df.error.notna()].groupby("method")
            .error.apply(lambda s: collections.Counter(e.split(":")[0] for e in s)))
print(summary, "\n\nrecorded failures and budget hits:\n", errors, sep="")

piv = ok.pivot_table(index=["dataset", "split"], columns="method", values="r2")
print("\nper-dataset means:\n", piv.groupby("dataset").mean().round(3))

try:
    st = json.load(open("results/split_stability.json"))
    for k, v in st.items():
        print(f"{k:14s} R2 {v['r2_mean']:.3f}\u00b1{v['r2_std']:.3f} "
              f"Jaccard(eq) {v['jaccard_mean']:.2f} stable classes {len(v['stable_features'])}")
except FileNotFoundError:
    pass

In [ ]:
!python make_figures.py